# Laboratório 7 — Fase "Dois Canais" (pseudo-binaural, ouvidos a 18 cm)
**Processamento de Sinais Multimédia** · PBL · 2026

Plano horizontal ($z = 0$): sala $L_x \times L_y$ com 4 paredes. O modelo é $y[n] = y_{direta}[n] + y_{reflexões}[n] + y_{reverberante}[n]$, calculado **para cada ouvido** (a $\pm 9$ cm do centro da cabeça), e o sinal anecoico é convolvido com a resposta impulsional de cada ouvido.

Parâmetros assumidos (célula seguinte): posições da fonte e do ouvinte, orientação da cabeça, $RT_{60}$ e a razão direta/reverberante. Substituir `RT60` pelo valor medido na aula.

In [ ]:
import numpy as np
import wave
import matplotlib.pyplot as plt
from IPython.display import Audio, display

plt.rcParams['figure.figsize'] = (12, 4.5)
plt.rcParams['axes.grid'] = True

# ------------------------------------------------------------------ parâmetros
Fs = 44100                      # frequência de amostragem (Hz)
c  = 343.0                      # velocidade do som (m/s)

Lx, Ly = 7.7, 9.4               # dimensões da sala (m), do guia
SRC    = np.array([2.5, 2.0])   # fonte (x, y) em m          <-- escolhida
LIS    = np.array([5.0, 6.5])   # centro da cabeça (x, y) em m <-- escolhida
FACING = np.array([0.0, -1.0])  # para onde o ouvinte olha (aqui, para a frente da sala, y = 0)
HEAD   = 0.18                   # distância entre ouvidos (m)
RT60   = 0.80                   # s  <-- PLACEHOLDER: usar o RT60 medido em sala
DRR_DB = -6.0                   # razão energia direta / energia da cauda (dB) <-- ajustável
BETA   = 0.35                   # sombra da cabeça: ganho do ouvido = 1 ± BETA·(componente lateral)
T_MIX  = 0.050                  # tempo de mistura (s), justificado na secção 4

# Filtros do trabalho anterior (TP1): atrasos dos Comb e All-Pass em ms
T_COMB_L = (29.7, 37.1, 41.1, 43.7)
T_AP_L   = (5.0, 1.7)
T_COMB_R = (31.1, 35.3, 42.3, 45.1)   # canal direito: atrasos diferentes -> cauda descorrelacionada
T_AP_R   = (5.3, 1.9)
A_AP     = 0.7

AUDIO_IN  = '../../psm-tp1-49774-51811/entrega/audio/in/voz_seca.wav'
AUDIO_OUT = 'Lab7_audio'

In [ ]:
def read_wav(path):
    # lê WAV PCM 16 bit para float mono em [-1, 1]
    with wave.open(path) as w:
        fs, ch = w.getframerate(), w.getnchannels()
        x = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16) / 32768.0
    if ch > 1:
        x = x.reshape(-1, ch).mean(axis=1)
    assert fs == Fs, f'fs = {fs}, esperado {Fs}'
    return x

def write_wav(path, x):
    # grava float [-1, 1] (mono ou N x 2) como PCM 16 bit
    pcm = np.round(np.clip(x, -1, 1) * 32767).astype(np.int16)
    with wave.open(path, 'wb') as w:
        w.setnchannels(1 if x.ndim == 1 else x.shape[1])
        w.setsampwidth(2); w.setframerate(Fs)
        w.writeframes(pcm.tobytes())

def fftconv(x, h):
    # convolução linear via FFT
    n = len(x) + len(h) - 1
    N = 1 << (n - 1).bit_length()
    return np.fft.irfft(np.fft.rfft(x, N) * np.fft.rfft(h, N), N)[:n]

def recursive_delay(u, N, g):
    # resolve y[n] = u[n] + g*y[n-N] por blocos de N amostras
    y = u.astype(float).copy()
    for s in range(N, len(y), N):
        e = min(s + N, len(y))
        y[s:e] += g * y[s - N:e - N]
    return y

def comb(x, N, g):
    # Comb: y[n] = x[n] + g*y[n-N]
    return recursive_delay(x, N, g)

def allpass(x, N, a):
    # All-Pass: y[n] = -a*x[n] + x[n-N] + a*y[n-N]
    v = -a * x
    v[N:] += x[:-N]
    return recursive_delay(v, N, a)

def edc_rt(h, t0=0):
    # Integração reversa de Schroeder; RT60 extrapolado de T20 e T30
    e = h[t0:] ** 2
    edc = np.cumsum(e[::-1])[::-1]
    d = 10 * np.log10(edc / edc[0] + 1e-30)
    t = np.arange(len(e)) / Fs
    out = {}
    for name, hi in (('T20', -25), ('T30', -35)):
        m = (d <= -5) & (d >= hi)
        out[name] = -60 / np.polyfit(t[m], d[m], 1)[0]
    return t, d, out

## 1. Definição espacial e geometria dos dois ouvidos
O ouvinte olha na direção `FACING`; o vetor para a sua esquerda é o de `FACING` rodado $+90°$. Os ouvidos ficam a $\pm 9$ cm do centro da cabeça ao longo desse eixo. O atraso interaural máximo possível é $0{,}18/343 \approx 0{,}52$ ms $\approx 23$ amostras.

In [ ]:
f = FACING / np.linalg.norm(FACING)
left = np.array([-f[1], f[0]])                 # esquerda do ouvinte
EAR_L = LIS + (HEAD / 2) * left
EAR_R = LIS - (HEAD / 2) * left

d0 = np.linalg.norm(SRC - LIS)
print(f'Distância fonte-centro da cabeça: {d0:.2f} m   atraso direto: {d0 / c * 1000:.1f} ms')
print(f'Ouvido esq. = {EAR_L.round(3)}   Ouvido dir. = {EAR_R.round(3)}')

## 2. Método das imagens (1.ª ordem, 4 paredes) para cada ouvido
A fonte-imagem de cada parede é o reflexo especular da fonte. Para **cada ouvido**, a distância $d_i$ à imagem dá o atraso $n_i = \mathrm{round}(d_i F_s / c)$ e o ganho

$$g_i = \frac{\sqrt{1-\bar\alpha}}{d_i}\,\bigl(1 \pm \beta\cos\theta_i\bigr)$$

O primeiro fator é a lei $1/d$ em amplitude (a intensidade cai com $1/d^2$) com a absorção da parede. O segundo é a **sombra da cabeça** simplificada: $\cos\theta_i$ é a componente da direção de chegada ao longo do eixo esquerda–direita ($+$ para o ouvido esquerdo, $-$ para o direito). Assim, uma reflexão que chega pela esquerda é mais forte no canal esquerdo.

Absorção a partir do $RT_{60}$ com a forma bidimensional da fórmula de Sabine (a altura cancela-se): $\bar\alpha = 0{,}161\,L_xL_y / \bigl(RT_{60}\cdot 2(L_x+L_y)\bigr)$.

In [ ]:
ALPHA = min(0.161 * Lx * Ly / (RT60 * 2 * (Lx + Ly)), 0.99)
R = np.sqrt(1 - ALPHA)
print(f'alfa médio = {ALPHA:.3f}   fator de amplitude por reflexão = {R:.3f}')

SURFACES = ['Parede x = 0', 'Parede x = Lx', 'Parede y = 0', 'Parede y = Ly']

def image_sources(S_pos):
    x, y = S_pos
    return [np.array(p) for p in ((-x, y), (2 * Lx - x, y), (x, -y), (x, 2 * Ly - y))]

IMAGES = image_sources(SRC)

def taps(rx, side=0):
    # chegadas (direta + 4 reflexões) num recetor rx; side = +1 esq., -1 dir., 0 sem sombra da cabeça
    out = []
    for name, pos, refl in [('Onda direta', SRC, 1.0)] + [(n, p, R) for n, p in zip(SURFACES, IMAGES)]:
        v = pos - rx
        d = np.linalg.norm(v)
        lateral = np.dot(v / d, left)                  # +1: chega pela esquerda
        out.append(dict(nome=name, d=d, t_ms=1000 * d / c, n=int(round(d / c * Fs)),
                        g0=refl / d, g=refl / d * (1 + BETA * side * lateral)))
    return out

tab_L, tab_R = taps(EAR_L, +1), taps(EAR_R, -1)
for lado, tab in (('OUVIDO ESQUERDO', tab_L), ('OUVIDO DIREITO', tab_R)):
    print(lado)
    print(f"{'Superfície':<16}{'d (m)':>8}{'t (ms)':>9}{'n (amostras)':>14}{'ganho (1/d·R)':>15}{'ganho c/ cabeça':>17}")
    for r in tab:
        print(f"{r['nome']:<16}{r['d']:>8.3f}{r['t_ms']:>9.2f}{r['n']:>14d}{r['g0']:>15.4f}{r['g']:>17.4f}")
    print()
print('Diferença interaural (esq - dir), em amostras:', [a['n'] - b['n'] for a, b in zip(tab_L, tab_R)])

with open(f'{AUDIO_OUT}/tabela_reflexoes_estereo.csv', 'w', encoding='utf-8') as fh:
    fh.write('ouvido,superficie,distancia_m,atraso_ms,atraso_amostras,ganho_1_sobre_d_x_R,ganho_com_cabeca\n')
    for lado, tab in (('esq', tab_L), ('dir', tab_R)):
        for r in tab:
            fh.write(f"{lado},{r['nome']},{r['d']:.4f},{r['t_ms']:.3f},{r['n']},{r['g0']:.5f},{r['g']:.5f}\n")

# planta
fig, ax = plt.subplots(figsize=(6, 6.5))
ax.add_patch(plt.Rectangle((0, 0), Lx, Ly, fill=False, lw=2))
ax.plot(*SRC, 'r*', ms=16, label='Fonte')
for im, n in zip(IMAGES, SURFACES):
    ax.plot(*im, 'x', color='gray'); ax.annotate(n.replace('Parede ', ''), im, fontsize=8, color='gray')
ax.plot(*EAR_L, 'bo', label='Ouvido esq.'); ax.plot(*EAR_R, 'gs', label='Ouvido dir.')
ax.annotate('', xy=LIS + 0.8 * f, xytext=LIS, arrowprops=dict(arrowstyle='->'))
ax.set_xlim(-Lx - 0.5, 2 * Lx + 0.5); ax.set_ylim(-Ly * 0.6, 2.6 * Ly); ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_title('Planta, fonte e fontes-imagem'); ax.legend(loc='upper right')
plt.show()

## 3. Campo reverberante, descorrelacionado entre canais
Reutiliza-se o reverberador de Schroeder do TP1 (4 *Comb* em paralelo + 2 *All-Pass* em série). Para o decaimento coincidir com o $RT_{60}$, cada *Comb* usa

$$g_k = 10^{-3 N_k / (F_s\, RT_{60})}$$

(queda de $-60$ dB em $RT_{60}$). **Descorrelação:** o canal direito usa atrasos de *Comb* e *All-Pass* diferentes dos do esquerdo, pelo que as duas caudas têm o mesmo decaimento mas formas de onda distintas (um campo difuso real é pouco correlacionado entre dois ouvidos).

**Energia.** A cauda tem energia $E_{cauda} = E_{direta}\cdot 10^{-DRR/10}$, com a mesma energia nos dois canais e `DRR_DB` ajustável. É um parâmetro porque, no plano horizontal, não há volume para calcular a distância crítica.

In [ ]:
def tail_ir(T_comb, T_ap, n_start, n_mix, n_ramp, E_target):
    # IR da cauda: Schroeder(δ) com fade-in a partir do tempo de mistura e energia E_target
    L = int((1.5 * RT60 + 0.3) * Fs)
    delta = np.zeros(L); delta[0] = 1.0
    y = np.zeros(L)
    for T in T_comb:
        N = round(T * Fs / 1000)
        y += comb(delta, N, 10 ** (-3 * N / (Fs * RT60)))
    for T in T_ap:
        y = allpass(y, round(T * Fs / 1000), A_AP)
    k = np.arange(L)
    w = np.ones(L)
    w[k < n_mix - n_ramp] = 0
    r = (k >= n_mix - n_ramp) & (k < n_mix)
    w[r] = 0.5 * (1 - np.cos(np.pi * (k[r] - (n_mix - n_ramp)) / n_ramp))
    y = y * w
    y *= np.sqrt(E_target / np.sum(y ** 2))
    h = np.zeros(n_start + L); h[n_start:] = y
    return h

n_mix, n_ramp = round(T_MIX * Fs), round(0.015 * Fs)
E_dir = tab_L[0]['g0'] ** 2
E_tail = E_dir * 10 ** (-DRR_DB / 10)
E_early = sum(r['g0'] ** 2 for r in tab_L[1:])
print(f'E_direta = {E_dir:.4f}   E_primeiras_reflexões = {E_early:.4f}   E_cauda = {E_tail:.4f}')

## 4. Tempo de mistura
É o instante em que as reflexões discretas dão lugar à cauda estatística. A estimativa empírica de Lindau et al., $t_{mix}\approx 0{,}0117\,V + 50{,}1$ ms, dá cerca de 52 ms para uma sala de aula típica (≈ 200 m³); a de Polack, $\sqrt{V}$ ms, dá cerca de 15 ms. Adotou-se **50 ms após a onda direta**: é posterior a todas as reflexões de 1.ª ordem (que aqui chegam nos primeiros 16 ms após a direta, ver tabela), pelo que ficam como eventos discretos, e a cauda entra com um *fade-in* de 15 ms para evitar uma descontinuidade.

In [ ]:
print('Última reflexão de 1.ª ordem após a direta: %.1f ms   (t_mix = %.0f ms)'
      % (max(r['t_ms'] for r in tab_L[1:]) - tab_L[0]['t_ms'], T_MIX * 1000))

## 5. Respostas impulsionais dos dois canais e processamento

In [ ]:
def early_ir(tab, n_len):
    h = np.zeros(n_len)
    for r in tab:
        h[r['n']] += r['g']
    return h

n_d0 = round(d0 / c * Fs)                     # a cauda parte da chegada da onda direta (centro da cabeça)
tail_L = tail_ir(T_COMB_L, T_AP_L, n_d0, n_mix, n_ramp, E_tail)
tail_R = tail_ir(T_COMB_R, T_AP_R, n_d0, n_mix, n_ramp, E_tail)
n_len = len(tail_L)
h_L = early_ir(tab_L, n_len) + tail_L
h_R = early_ir(tab_R, n_len) + tail_R

seg = slice(n_d0 + n_mix, n_len)
print(f'Correlação das caudas L/R (após t_mix): {np.corrcoef(tail_L[seg], tail_R[seg])[0, 1]:.3f}')
w40 = slice(n_d0 - 5, n_d0 + round(0.04 * Fs))
print(f'Energia nos primeiros 40 ms: esq. {np.sum(h_L[w40] ** 2):.4f}   dir. {np.sum(h_R[w40] ** 2):.4f}')

t = (np.arange(n_len) - n_d0) / Fs * 1000
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, h, nome in ((a1, h_L, 'Esquerdo'), (a2, h_R, 'Direito')):
    ax.plot(t, 20 * np.log10(np.abs(h) + 1e-9), lw=0.5)
    ax.axvline(0, color='r', ls='--', label='direta'); ax.axvline(T_MIX * 1000, color='k', ls=':', label='t_mix')
    ax.set_xlim(-5, 300); ax.set_ylim(-110, 0); ax.set_xlabel('t após a direta (ms)'); ax.set_ylabel('dB')
    ax.set_title(f'Resposta impulsional, ouvido {nome}'); ax.legend()
plt.tight_layout(); plt.show()

# ampliação das primeiras reflexões nos dois ouvidos
fig, ax = plt.subplots()
m = (t > -2) & (t < 25)
ax.stem(t[m] + 0, h_L[m], linefmt='b-', markerfmt='bo', basefmt=' ', label='Esq.')
ax.stem(t[m] + 0, -h_R[m], linefmt='g-', markerfmt='gs', basefmt=' ', label='Dir. (espelhado)')
ax.set_xlabel('t após a direta (ms)'); ax.set_title('Direta e primeiras reflexões nos dois ouvidos'); ax.legend(); plt.show()

In [ ]:
x = read_wav(AUDIO_IN)
print(f'Áudio anecoico: {len(x) / Fs:.2f} s, pico {np.max(np.abs(x)):.2f}')

y_st = np.stack([fftconv(x, h_L), fftconv(x, h_R)], axis=1)
print(f'Pico antes de normalizar: {np.max(np.abs(y_st)):.3f}')
y_st_n = 0.99 * y_st / np.max(np.abs(y_st))     # fator comum aos 2 canais: preserva a diferença interaural de nível
x_n = 0.99 * x / np.max(np.abs(x))

# referência mono: média das duas respostas (sem informação interaural)
y_mono_n = 0.99 * fftconv(x, 0.5 * (h_L + h_R)) / np.max(np.abs(fftconv(x, 0.5 * (h_L + h_R))))

write_wav(f'{AUDIO_OUT}/1_original_anecoico.wav', x_n)
write_wav(f'{AUDIO_OUT}/2_referencia_mono.wav', y_mono_n)
write_wav(f'{AUDIO_OUT}/3_processado_estereo.wav', y_st_n)
print('Pico final estéreo: %.3f   amostras >= 0.999: %d' % (np.max(np.abs(y_st_n)), np.sum(np.abs(y_st_n) >= 0.999)))

print('original'); display(Audio(x_n, rate=Fs))
print('estéreo (usar auscultadores)'); display(Audio(y_st_n.T, rate=Fs))

fig, axs = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, sig, nome in zip(axs, (x_n, y_st_n[:, 0], y_st_n[:, 1]), ('Original', 'Canal esq.', 'Canal dir.')):
    ax.plot(np.arange(len(sig)) / Fs, sig, lw=0.4); ax.set_ylabel(nome); ax.set_ylim(-1.05, 1.05)
axs[-1].set_xlabel('t (s)'); plt.tight_layout(); plt.show()

## 6. Validação
1. $RT_{60}$ de cada canal por integração reversa (como em `RT60Estimacao.py`) contra o alvo.
2. Clipping: ver pico final acima.
3. Fonte e ouvinte em várias posições: diferença interaural de tempo (ITD) e de nível da onda direta.

In [ ]:
def rt_of(h, t0):
    e = h[t0:] ** 2
    edc = np.cumsum(e[::-1])[::-1]; d = 10 * np.log10(edc / edc[0] + 1e-30); tt = np.arange(len(e)) / Fs
    return tt, d, {k: -60 / np.polyfit(tt[(d <= -5) & (d >= hi)], d[(d <= -5) & (d >= hi)], 1)[0] for k, hi in (('T20', -25), ('T30', -35))}

fig, ax = plt.subplots()
for h, nome in ((h_L, 'Esq.'), (h_R, 'Dir.')):
    tt, d, rt = rt_of(h, n_d0)
    ax.plot(tt, d, label=f'{nome}: T20 = {rt["T20"]:.2f} s, T30 = {rt["T30"]:.2f} s')
ax.plot([0, RT60], [0, -60], 'k--', label=f'alvo RT60 = {RT60:.2f} s')
ax.set_ylim(-70, 3); ax.set_xlim(0, 1.5); ax.set_xlabel('t (s)'); ax.set_ylabel('EDC (dB)'); ax.legend()
ax.set_title('Decaimento de energia (integração reversa)'); plt.show()

In [ ]:
POSICOES = [('Fundo, centro',   np.array([5.0, 6.5])),
            ('Meio da sala',    np.array([4.0, 5.0])),
            ('Perto da fonte',  np.array([2.5, 3.5])),
            ('Junto à parede x=Lx', np.array([7.2, 4.0])),
            ('Canto traseiro',  np.array([7.0, 9.0]))]

print(f"{'Posição':<22}{'d (m)':>7}{'ITD (amostras)':>16}{'ILD direta (dB)':>17}{'E_dir/(E_refl+E_cauda) dB':>27}")
for nome, p in POSICOES:
    ll = np.array([-f[1], f[0]])
    eL, eR = p + HEAD / 2 * ll, p - HEAD / 2 * ll
    v = SRC - p; d = np.linalg.norm(v)
    tl, tr = taps(eL, +1), taps(eR, -1)
    itd = tl[0]['n'] - tr[0]['n']
    ild = 20 * np.log10(tl[0]['g'] / tr[0]['g'])
    E_ref = sum(r['g0'] ** 2 for r in tl[1:])
    print(f'{nome:<22}{d:>7.2f}{itd:>16d}{ild:>17.2f}{10 * np.log10((1 / d ** 2) / (E_ref + E_tail)):>27.1f}')

## 7. Análise crítica e limitações
*(Esboço: confirmar com a escuta e com os resultados da execução, e adaptar.)*

**O que o modelo capta.** Para cada ouvido: a onda direta com atraso e atenuação por distância, as 4 reflexões de 1.ª ordem com atrasos e ganhos da geometria, e uma cauda com o $RT_{60}$ da sala. As diferenças de tempo (até 23 amostras) e de nível entre ouvidos, mais a descorrelação das caudas, dão a sensação de espacialidade e de largura da sala.

**Limitações.**
- **Plano horizontal:** sem chão nem teto não há reflexões verticais, que numa sala real chegam muito cedo (aqui, de chão e teto, cerca de 2 ms após a direta) e influenciam a cor e a elevação percebida. A absorção usa a forma 2D de Sabine, que ignora chão e teto.
- Só reflexões de **1.ª ordem** são calculadas; as restantes ficam a cargo da cauda estatística.
- **Sombra da cabeça simplificada** ($1\pm\beta\cos\theta$) e atrasos por geometria: falta o filtro de cada ouvido (HRTF: pavilhão, cabeça e tronco, dependentes da frequência e da direção). Resultado: localização mais fraca, sons percebidos "dentro da cabeça" e confusão frente/trás.
- Absorção **igual para todas as paredes e todas as frequências**; numa sala real o $RT_{60}$ varia com a frequência.
- Atrasos **arredondados à amostra** (erro até 11 µs), com algum efeito nas diferenças interaurais pequenas (resolução de 1 amostra ≈ 7,8 mm de percurso).
- A cauda de Schroeder tem coloração e menor densidade de ecos que uma sala real; a descorrelação por atrasos distintos é eficaz, mas não replica a coerência dependente da frequência de um campo difuso real (alta nos graves).
- Fonte omnidirecional, sala vazia e superfícies puramente especulares; `RT60`, `DRR_DB` e as posições são parâmetros assumidos.